# Wild-Diff-ICMH — H1 Recovery và tiếp tục H2 trong cùng phiên

**Đã chạy xong H1 Recovery 200 và chấm 30 ảnh?** Giữ nguyên notebook này,
phiên L4 và dữ liệu `/content`. Không chạy lại phần training H1. Dùng phần
**H2 tiếp tục trong phiên hiện tại** ở cuối notebook: chuẩn bị/xem ROI → tự
xác nhận → train H2 200 → chấm riêng H2 30 ảnh → gửi kết quả. Đây là một
lượt thử ưu tiên vùng con vật, không phải tái hiện Xie-SGC hoặc quét cấu hình.
Nội dung H1 bên dưới được giữ để tái lập; các cờ nặng mặc định vẫn TẮT.

**H1 cũ đã tới 1.000 bước.** Giữ nguyên run, checkpoint và kết quả đó.
Notebook này đọc dữ liệu 30 ảnh đã lưu trước; **chạy 1 → 2 → 3 bằng CPU**.
Không cần GPU, chép toàn bộ ảnh, mở checkpoint nhiều gigabyte hay cài detector để đọc báo cáo.

Chỉ khi báo cáo cho phép và bạn đồng ý mới bật thử nghiệm ở Bước 4, chuyển runtime
sang **L4**, rồi chạy lại **1 → 4 → 5 → 6 → 7**. Run mới có tối đa **200 bước tổng**,
khởi đầu từ trọng số tác giả, optimizer mới; tốc độ học chính 0,00001,
phụ 0,001. Đây là giả thuyết cần kiểm tra, không phải lỗi tốc độ học đã được xác nhận.
Sau đó bật riêng Bước 8 để đánh giá đúng 30 ảnh; Bước 9 đọc kết quả và quyết định.

**Run all mặc định không train hoặc sinh ảnh.** Không chạy lại B0, Xie-SGC, 3.000 bước,
toàn bộ validation hay test. Phase 3 chưa hoàn thành chỉ vì notebook chạy hết.
Sửa script sau đó dùng **2A** rồi chạy lại cell lỗi. Pull không thay cấu trúc
notebook đang mở. Nếu notebook mở trước bản H2, chỉ thêm một cell continuation
ở cuối **chính notebook/phiên này** theo `docs/H2_ROI_PILOT.md`; các lần sửa sau
dùng pull và chạy lại cell ấy. Không mở notebook khác để đổi luồng.

## 1 — Gắn Drive, chọn đường dẫn (CPU được)

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/wild_diff_icmh')
CU_RATE = 1.54  # dùng số CU/giờ Colab đang báo, chỉ là ước tính
CU_AVAILABLE_AT_START = None  # điền Available để đo chi phí thật cả phiên
REVIEW_PATH = DRIVE_ROOT / 'phase3/h1_low_lr_recovery/review.json'
REVIEW_MD = REVIEW_PATH.with_suffix('.md')
RUN_DIR = DRIVE_ROOT / 'runs/h1_low_lr_recovery_ls512/lambda_2'
RECOVERY_ARTIFACTS = DRIVE_ROOT / 'phase3/h1_low_lr_recovery'


## 2 — Lấy main, không yêu cầu GPU

In [ ]:
REPO_URL = 'https://github.com/TranDuon/Wild-Diff-ICMH.git'
BRANCH = 'main'
REPO = Path('/content/Wild-Diff-ICMH')

def run_checked(command, cwd=None, check=True, **kwargs):
    command = list(map(str, command))
    with subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, **kwargs,
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    if check and return_code:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code

def safe_pull():
    current_branch = subprocess.check_output(
        ['git', 'branch', '--show-current'], cwd=REPO, text=True,
    ).strip()
    assert current_branch == BRANCH, f'Đang ở nhánh {current_branch}, không tự đổi nhánh hay ghi đè code.'
    local_changes = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO, text=True,
    ).strip()
    if local_changes:
        raise RuntimeError('Code có thay đổi cục bộ; giữ nguyên và gửi log để xử lý:\n' + local_changes)
    before = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    run_checked(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO)
    after = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    remote = subprocess.check_output(['git', 'rev-parse', f'origin/{BRANCH}'], cwd=REPO, text=True).strip()
    assert after == remote, f'Commit local {after} chưa khớp GitHub {remote}'
    print('Commit trước:', before[:12], 'sau:', after[:12])
    print('PULL THÀNH CÔNG — chạy lại đúng cell vừa lỗi.', flush=True)

if (REPO / '.git').is_dir():
    safe_pull()
else:
    run_checked(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)])
os.chdir(REPO)
print('Mã nguồn:', REPO)

def recovery(action, *options):
    # A new child uses the latest pulled script, not a stale imported module.
    command = [sys.executable, '-u', 'tools/phase3_h1_recovery.py', action,
               '--drive-root', str(DRIVE_ROOT), '--cu-rate', str(CU_RATE), *map(str, options)]
    print('$', ' '.join(command), flush=True)
    with subprocess.Popen(
        command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## 2A — Pull cố định sau khi có bản sửa đã push

In [ ]:
assert (REPO / '.git').is_dir(), 'Chạy Bước 2 trước.'
safe_pull()


## 3 — Đọc và tính lại kết quả đã lưu trên đúng 30 ảnh (CPU)

Chỉ cần gói chấm điểm nhỏ `pycocotools`; không cài mô hình học sâu hay tạo lại ảnh.
Báo cáo so B0 ở ba mức nén với H1 500/1.000 bước trên cùng danh sách ảnh.
Dữ liệu/protocol thiếu hoặc khác nhau sẽ dừng, không lặng lẽ ghép các kết quả.
Nếu thiếu ảnh để nhìn hoặc log thành phần loss, báo cáo ghi rõ chưa kiểm tra được.

In [ ]:
import importlib.util
if importlib.util.find_spec('pycocotools') is None:
    run_checked([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'pycocotools==2.0.11'])
recovery('review')
recovery('status')
print('Báo cáo:', REVIEW_PATH)
if REVIEW_MD.is_file():
    from IPython.display import Markdown, display
    display(Markdown(REVIEW_MD.read_text(encoding='utf-8')))


## 4 — Quyết định và ngân sách: tất cả hành động nặng đang TẮT

Đọc báo cáo trước. Nếu được chọn thử, đổi `RUN_TRIAL = True` và
`ACKNOWLEDGE_REVIEW = REVIEW_SHA256` (không cần copy đoạn code khác).
Không bật đánh giá cùng lúc nếu chưa train đủ 200 bước. Mỗi runtime mới chạy 1–4 lại.
Nếu hash thay đổi, phải đọc báo cáo mới và xác nhận lại, không dùng quyết định cũ.

Ước lượng **riêng training**: 200 × 5,65 giây ≈ 18,8 phút, khoảng 0,48 CU
nếu mức tiêu thụ là 1,54 CU/giờ; **chưa gồm setup, nạp mô hình, checkpoint hay đánh giá**.
Có thể dừng theo thời gian rồi chạy lại Bước 7; chỉ resume run mới, không học lại từ đầu.

In [ ]:
assert REVIEW_PATH.is_file(), 'Chạy Bước 3 để tạo báo cáo trước.'
REVIEW_SHA256 = hashlib.sha256(REVIEW_PATH.read_bytes()).hexdigest()
print('Hash báo cáo cần xác nhận:', REVIEW_SHA256)
RUN_TRIAL = False
RUN_EVALUATION = False
ACKNOWLEDGE_REVIEW = ''  # sau khi đọc báo cáo: đổi thành REVIEW_SHA256
SESSION_HOURS = 0.5  # 0 < số giờ <= 2; mỗi lượt train, không gồm setup

def require_review_acknowledgement():
    current_hash = hashlib.sha256(REVIEW_PATH.read_bytes()).hexdigest()
    assert current_hash == REVIEW_SHA256, 'Báo cáo đã đổi; đọc lại Bước 3 → 4.'
    assert ACKNOWLEDGE_REVIEW == current_hash, 'Chưa xác nhận đã đọc đúng báo cáo hiện tại.'
    review = json.loads(REVIEW_PATH.read_text(encoding='utf-8'))
    assert review.get('training_allowed') is True, 'Báo cáo chưa cho phép thử; xem lý do trước.'
    assert review.get('protocol_verified') is True, 'Chưa xác nhận cùng protocol; không mở hành động nặng.'
    assert review.get('recommendation') == 'conditional_low_lr_trial', 'Quyết định không phải lượt thử tốc độ học thấp.'
    assert 0 < SESSION_HOURS <= 2, 'Giới hạn phiên phải lớn hơn 0 và không quá 2 giờ.'

print('Run mới:', RUN_DIR)
print('Artifacts riêng:', RECOVERY_ARTIFACTS)
print('Chỉ đọc kết quả: dừng tại đây, không cần GPU.')


## 5 — Chỉ khi đã bật thử/đánh giá: kiểm tra GPU và môi trường đã dùng

Chọn L4 rồi **Lưu** cấu hình runtime và kết nối phiên mới. Nếu chưa có GPU thực,
cell dừng trước khi cài môi trường; không chỉ bỏ qua lỗi rồi train trên CPU.
Giữ Torch/NumPy/SciPy của Colab, dùng quy trình cài đặt đã kiểm tra của dự án.

In [ ]:
if RUN_TRIAL or RUN_EVALUATION:
    require_review_acknowledgement()
    assert shutil.which('nvidia-smi'), 'Phiên này chưa có GPU. Chọn L4, Lưu, kết nối lại rồi chạy 1–4.'
    run_checked(['nvidia-smi'])
    source_requirements = REPO / 'requirements-colab.txt'
    safe_requirements = Path('/content/requirements-colab-safe.txt')
    core_constraints = Path('/content/colab-core-constraints.txt')

    from importlib import metadata as importlib_metadata

    core_distributions = ('numpy', 'scipy', 'torch', 'torchvision')
    core_versions_before = {
        name: importlib_metadata.version(name)
        for name in core_distributions
    }

    filtered_lines = []
    for line in source_requirements.read_text(encoding='utf-8').splitlines():
        normalized = line.strip().lower()
        if normalized.startswith(('numpy', 'scipy')):
            print('Giữ bản Colab, bỏ qua:', line)
            continue
        filtered_lines.append(line)

    safe_requirements.write_text(
        '\n'.join(filtered_lines) + '\n',
        encoding='utf-8',
    )

    core_constraints.write_text(
        '\n'.join(
            f'{name}=={version}'
            for name, version in core_versions_before.items()
        ) + '\n',
        encoding='utf-8',
    )

    run_checked([
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', str(safe_requirements),
        '-c', str(core_constraints),
    ], check=True)
    run_checked([
        sys.executable, '-m', 'pip', 'install', '-q',
        '--no-deps', '--no-build-isolation',
        'compressai==1.2.8',
    ], check=True)
    run_checked([
        sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
        '--force-reinstall', 'src/recognize-anything',
    ], cwd=REPO, check=True)

    import importlib
    import numpy as np
    import numpy.random as npr
    import scipy
    import torch
    import torch_geometric
    import pytorch_msssim
    import compressai.ans
    import compressai.entropy_models
    import compressai.losses
    import compressai.zoo

    # A pip subprocess can add files after this Jupyter kernel initialized.
    # Refresh import caches so the freshly installed vendored package is
    # visible immediately, without requiring a runtime restart.
    importlib.invalidate_caches()
    import ram
    from model.lfgcm import TagGCM

    core_versions_after = {
        name: importlib_metadata.version(name)
        for name in core_distributions
    }
    assert core_versions_after == core_versions_before, (
        'Dependency install changed Colab core packages: '
        f'before={core_versions_before}, after={core_versions_after}'
    )
    assert torch.cuda.is_available(), 'Vào Runtime > Change runtime type > chọn GPU'
    gpu = torch.cuda.get_device_properties(0)
    print('Python:', sys.version.split()[0])
    print('NumPy:', np.__version__, np.__file__)
    print('SciPy:', scipy.__version__)
    print('Torch:', torch.__version__, torch.version.cuda)
    print('PyG:', torch_geometric.__version__)
    print('RAM package:', ram.__file__)
    print('Core packages preserved:', core_versions_after)
    print('GPU:', gpu.name, f'{gpu.total_memory / 2**30:.1f} GiB')
    print('Kiểm tra numpy.random:', npr.rand(3))
    print('THÀNH CÔNG: môi trường và TagGCM đã sẵn sàng.')
else:
    print('Bỏ qua: chưa bật hành động GPU này.')


## 6 — Chỉ sau khi đồng ý: chép tài sản training đã có (không tạo tags/B0)

Chép ảnh và checkpoint từ Drive vào `/content` có thể mất thời gian.
Tự bỏ qua file đã đủ; chỉ chép checkpoint Stable Diffusion và tác giả λ = 2.
Không tải/chép RAM++ weights, vì dùng bộ tags đã lưu. Không chạm run H1 cũ.

In [ ]:
if RUN_TRIAL or RUN_EVALUATION:
    require_review_acknowledgement()
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm.auto import tqdm

    # DRIVE_ROOT đã chọn ở Bước 1; không đổi đường dẫn giữa chừng.
    DRIVE_IMAGES = DRIVE_ROOT / 'images' / 'snapshot_kgalagadi'
    LOCAL_IMAGES = Path('/content/data/wild_diff_icmh/images')
    LOCAL_SITE_IMAGES = LOCAL_IMAGES / 'snapshot_kgalagadi'

    assert DRIVE_IMAGES.is_dir(), f'Không thấy dữ liệu: {DRIVE_IMAGES}'
    LOCAL_SITE_IMAGES.mkdir(parents=True, exist_ok=True)

    print('Đang quét file trên Drive...')
    source_files = [path for path in DRIVE_IMAGES.rglob('*') if path.is_file()]
    pending = []
    for src in source_files:
        dst = LOCAL_SITE_IMAGES / src.relative_to(DRIVE_IMAGES)
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            pending.append((src, dst, src.stat().st_size))

    remaining_bytes = sum(size for _, _, size in pending)
    print('Tổng file:', len(source_files))
    print('Còn phải copy:', len(pending), 'file')
    print('Dung lượng còn lại:', f'{remaining_bytes / 2**30:.2f} GiB')

    def copy_one(item):
        src, dst, size = item
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + '.part')
        try:
            with src.open('rb') as fin, tmp.open('wb') as fout:
                while chunk := fin.read(8 * 1024 * 1024):
                    fout.write(chunk)
            os.replace(tmp, dst)
            return size
        except Exception:
            if tmp.exists():
                tmp.unlink()
            raise

    if pending:
        with ThreadPoolExecutor(max_workers=6) as executor:
            futures = [executor.submit(copy_one, item) for item in pending]
            with tqdm(
                total=remaining_bytes,
                unit='B', unit_scale=True, unit_divisor=1024,
                desc='Copy Kgalagadi',
            ) as progress:
                for future in as_completed(futures):
                    progress.update(future.result())
    else:
        print('Ảnh cục bộ đã đầy đủ, không cần copy lại.')

    run_checked([
        sys.executable, 'tools/data/split_check.py',
        'data/manifests/kgalagadi_site_split.jsonl',
        '--build-info', 'data/manifests/build_info.json',
    ], cwd=REPO, check=True)
    print('Ảnh và manifest đã sẵn sàng.')


    DRIVE_CKPT_ROOT = DRIVE_ROOT / 'checkpoints'
    CKPT_ROOT = Path('/content/data/wild_diff_icmh/checkpoints')
    BPP_WEIGHT = 2
    folder = f'CNscale1.0_1_1_{BPP_WEIGHT}_2_WTagGCM_bs16x1_lr0.00005_cfg7.0'

    checkpoint_relatives = [
        Path('sd2p1/v2-1_512-ema-pruned.ckpt'),
        Path('difficmh_models') / folder / 'model.ckpt',
    ]
    backup_items = []
    for relative in checkpoint_relatives:
        src = DRIVE_CKPT_ROOT / relative
        dst = CKPT_ROOT / relative
        assert src.is_file(), f'Thiếu checkpoint trên Drive: {src}'
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            backup_items.append((src, dst, src.stat().st_size))

    total_bytes = sum(size for _, _, size in backup_items)
    if backup_items:
        def copy_checkpoint(item, progress):
            src, dst, _ = item
            dst.parent.mkdir(parents=True, exist_ok=True)
            tmp = dst.with_name(dst.name + '.part')
            try:
                with src.open('rb') as fin, tmp.open('wb') as fout:
                    while chunk := fin.read(8 * 1024 * 1024):
                        fout.write(chunk)
                        progress.update(len(chunk))
                os.replace(tmp, dst)
                return dst
            except Exception:
                if tmp.exists():
                    tmp.unlink()
                raise

        with tqdm(
            total=total_bytes,
            unit='B', unit_scale=True, unit_divisor=1024,
            desc='Khôi phục checkpoint',
        ) as progress:
            with ThreadPoolExecutor(max_workers=2) as executor:
                futures = [
                    executor.submit(copy_checkpoint, item, progress)
                    for item in backup_items
                ]
                for future in as_completed(futures):
                    print('\nĐã khôi phục:', future.result())
    else:
        print('Checkpoint cục bộ đã đầy đủ, không cần copy lại.')

    AUTHOR_CKPT = CKPT_ROOT / 'difficmh_models' / folder / 'model.ckpt'
    repo_checkpoints = REPO / 'checkpoints'
    if not repo_checkpoints.exists():
        repo_checkpoints.symlink_to(CKPT_ROOT, target_is_directory=True)
    elif repo_checkpoints.resolve() != CKPT_ROOT.resolve():
        raise RuntimeError(f'{repo_checkpoints} không trỏ tới {CKPT_ROOT}')

    print('Diff-ICMH:', AUTHOR_CKPT)
    print('Checkpoint đã sẵn sàng.')
else:
    print('Bỏ qua: chưa bật hành động GPU này.')


## 7 — Thử H1 tốc độ học thấp: tối đa 200 bước tổng (TẮT mặc định)

Lần đầu nạp trọng số tác giả, không nạp optimizer H1 cũ. Lần sau chỉ tiếp tục
checkpoint có hợp đồng và tốc độ học đúng của run mới. Đủ 200 sẽ không train thêm.
Trong training chỉ tính loss, không sinh ảnh; ghi thành phần loss và tốc độ học
để lần này có dữ liệu kiểm tra. Không đổi λ, crop, dữ liệu hay trọng số loss cùng lúc.

In [ ]:
if RUN_TRIAL:
    require_review_acknowledgement()
    recovery('train', '--enable', '--acknowledge-review', ACKNOWLEDGE_REVIEW,
             '--session-hours', SESSION_HOURS)
    recovery('status')
else:
    print('Bỏ qua train: RUN_TRIAL vẫn False.')


## 8 — Đánh giá riêng checkpoint mới trên 30 ảnh (TẮT mặc định)

Sau khi đủ 200 bước, đổi `RUN_TRIAL = False`, `RUN_EVALUATION = True` ở Bước 4
rồi chạy lại 4 → 5 → 6 → 8 → 9. Detector chỉ cài tại đây; không decode/chạy lại B0.
Cùng cạnh dài 512, 50 bước lấy mẫu, seed 231, mức hướng dẫn 3, chấm ảnh ở kích thước gốc.
Chi phí đánh giá ngoài ước tính training 0,48 CU phía trên.

In [ ]:
if RUN_EVALUATION:
    require_review_acknowledgement()
    def run_logged(command, log_name, echo=True):
        log_path = RECOVERY_ARTIFACTS / 'logs' / log_name
        log_path.parent.mkdir(parents=True, exist_ok=True)
        command = list(map(str, command))
        with log_path.open('a', encoding='utf-8') as log_stream:
            with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                  text=True, bufsize=1) as process:
                for line in process.stdout:
                    if echo:
                        print(line, end='', flush=True)
                    log_stream.write(line)
                    log_stream.flush()
                return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)
    # Colab's Python has no ensurepip, so ``python -m venv`` cannot install pip;
    # virtualenv ships its own.  --system-site-packages reuses Colab's torch.
    DETECT_ENV = Path('/content/envs/h1-recovery-detect')
    DETECT_PY = DETECT_ENV / 'bin' / 'python'
    DETECT_READY = DETECT_ENV / '.ready'
    if not DETECT_READY.exists():
        DETECT_ENV.mkdir(parents=True, exist_ok=True)  # no deletion of another environment
        run_logged([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'], 'h1_recovery_detect_env.log')
        run_logged([sys.executable, '-m', 'virtualenv', '--system-site-packages', DETECT_ENV], 'h1_recovery_detect_env.log')
        # The env also sees Colab's transformers, which needs Colab's
        # huggingface-hub (<1.0); PytorchWildlife's deps would otherwise pull 1.x.
        import importlib.metadata
        hub_pin = f"huggingface-hub=={importlib.metadata.version('huggingface_hub')}"
        run_logged([DETECT_PY, '-m', 'pip', 'install', '-q', 'PytorchWildlife', hub_pin], 'h1_recovery_detect_env.log')
        run_logged([DETECT_PY, '-c', 'from PytorchWildlife.models import detection; print("PytorchWildlife OK")'],
                   'h1_recovery_detect_env.log')
        DETECT_READY.touch()

    recovery('evaluate', '--enable', '--acknowledge-review', ACKNOWLEDGE_REVIEW,
             '--detect-python', DETECT_PY)
else:
    print('Bỏ qua: chưa bật hành động GPU này.')


## 9 — Đọc status/kết quả mới và gửi lại để quyết định (CPU)

Gửi `review.json`, kết quả comparison mới, `throughput.json` và `loss_components.csv`
trong run mới nếu đã train. Không kết luận mô hình tốt hơn chỉ từ 30 ảnh hoặc từ
điểm có dung lượng khác nhau. Không tự kéo dài training, đánh giá rộng hoặc bật H2.

In [ ]:
recovery('status')
comparisons = sorted((RECOVERY_ARTIFACTS / 'eval').glob('**/comparison.json'))
for path in comparisons:
    print('Kết quả mới:', path)
    comparison = json.loads(path.read_text(encoding='utf-8'))
    for method, values in comparison.get('points', {}).items():
        quality = values.get('quality', {})
        machine = values.get('machine', {})
        print(method, 'số ảnh:', quality.get('n'), 'bit/pixel:', quality.get('bpp'))
        print('  chất lượng gần gốc:', quality.get('psnr'), 'sai khác cảm nhận:', quality.get('lpips'))
        print('  điểm phát hiện:', machine.get('map'), 'ảnh bỏ sót:', machine.get('missed_animal_rate'),
              'phát hiện giả:', machine.get('hallucination_rate'))
    print('Thông số đã xác minh:', comparison.get('protocol_verified'))
    print('Quyết định tham khảo:', comparison.get('recommendation'))
    loss_review = comparison.get('loss_review', {})
    print('Log thành phần loss:', loss_review.get('status'))
    for missing in loss_review.get('missing', []):
        print('  Chưa có bằng chứng:', missing)
print('Báo cáo CPU:', REVIEW_PATH)
print('Throughput nếu đã train:', RUN_DIR / 'throughput.json')
print('Loss/tốc độ học nếu đã train:', RUN_DIR / 'loss_components.csv')
print('Phase 3 chưa tự đánh dấu hoàn thành; H2 là bước thiết kế tiếp theo.')


## 10 — Kết thúc phiên; chỉ chủ động bật khi đã xong

Nếu đang giữ L4 mà không làm gì nữa, ngắt runtime để không tiêu thêm CU.
Checkpoint và báo cáo đã ghi trên Drive còn; file `/content` có thể mất.

In [ ]:
CU_AVAILABLE_AT_END = None
if CU_AVAILABLE_AT_START is not None and CU_AVAILABLE_AT_END is not None:
    print('CU thật cả phiên:', CU_AVAILABLE_AT_START - CU_AVAILABLE_AT_END)
DISCONNECT_NOW = False
if DISCONNECT_NOW:
    from google.colab import runtime
    drive.flush_and_unmount()
    runtime.unassign()


## H2 — Tiếp tục trong phiên L4 hiện tại, không tạo notebook/runtime mới

Chỉ dùng khi H1 Recovery đã train **200/200** và đã có kết quả chấm 30 ảnh.
Không chạy lại cell H1 4–8. Cell dưới chỉ hiển thị bảng nút; **chưa chép ảnh,
cài môi trường, train hay sinh ảnh**. Lần lượt: chuẩn bị/xem ROI → tick xác nhận
→ train H2 tới 200 → chấm H2 30 ảnh → đọc trạng thái.

Run H2 riêng, khởi đầu cùng trọng số tác giả/optimizer mới như H1; chỉ đổi
trọng số vùng con vật từ 1 lên 2. H1/B0 đã chấm được giữ nguyên và dùng lại.
Ảnh local đủ dung lượng được bỏ qua; đánh giá chỉ chuẩn bị đúng 30 ảnh.
Không reset hoặc ngắt L4 đang có. Mỗi lần sửa: pull **2A** rồi chạy lại cell
này để dùng giao diện/script mới; không cần copy thêm cell sửa lỗi.

In [ ]:
# Dùng đúng REPO/DRIVE_ROOT của Recovery đang mở, giữ toàn bộ /content.
RUN_TRIAL = False
RUN_EVALUATION = False
safe_pull()
import runpy
H2_CONTINUATION = runpy.run_path(
    str(REPO / 'tools/colab_h2_continuation.py'), run_name='__main__',
    init_globals={'REPO': REPO, 'DRIVE_ROOT': DRIVE_ROOT, 'CU_RATE': CU_RATE},
)
